# BioNER — BioBERT for Biomedical NER (EMBO SourceData)

Token-classification baseline for the EMBO SourceData NER task: **BioBERT (`dmis-lab/biobert-base-cased-v1.2`)
+ linear token classifier**, trained with cross-entropy and strict IOB2 (seqeval) scoring.

**Best validation result (this notebook, final scheme):** P 0.829 / R 0.850 / **F1 0.8391** @ 3 epochs.

Run history (all on validation, strict IOB2):
| run | label-alignment scheme | val F1 |
|---|---|---|
| v1 | continuation subwords get `I-` (propagation) | 0.8323 |
| v2 | continuation subwords masked `-100` (**kept**) | 0.8391 |
| v3 | BioBERT + CRF + Dice loss (separate notebook) | 0.8373 (argmax eval — see notes §11) |

The `-100` masking scheme won the ablation, so it's the one implemented here.

## 1. Setup

`evaluate` + `seqeval` for sequence-level (entity-level) metrics — plain accuracy is misleading for NER.

In [ ]:
!pip install -U pip setuptools wheel
!pip install evaluate seqeval


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 kB 41.8 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 80.10.2
    Uninstalling setuptools-80.10.2:
      Successfully uninstalled setuptools-80.10.2
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cu128 requires setuptools<82, but you have setuptools 84.0.0 which is incompatible.


  Installing build dependencies ... canceled
ERROR: Operation cancelled by user
^C


## 2. Load dataset

Three splits from the EMBO SourceData HF repo, in JSONL token-classification format (`words`, `labels`, plus `is_category`/`text` metadata).

In [1]:
from datasets import load_dataset

data_files = {
    "train": "https://huggingface.co/datasets/EMBO/SourceData/resolve/main/token_classification/ner/train.jsonl",
    "validation": "https://huggingface.co/datasets/EMBO/SourceData/resolve/main/token_classification/ner/validation.jsonl",
    "test": "https://huggingface.co/datasets/EMBO/SourceData/resolve/main/token_classification/ner/test.jsonl",
}

ds = load_dataset("json", data_files=data_files)
print(ds)


token_classification/ner/train.jsonl: reconstructing file:   0%|          |  0.00B /  104MB            

token_classification/ner/train.jsonl: downloading bytes:           |  0.00B            

token_classification/ner/validation.json(…): reconstructing file:   0%|          |  0.00B / 14.7MB            

token_classification/ner/validation.json(…): downloading bytes:           |  0.00B            

token_classification/ner/test.jsonl: reconstructing file:   0%|          |  0.00B / 13.3MB            

token_classification/ner/test.jsonl: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['words', 'labels', 'is_category', 'text'],
        num_rows: 55250
    })
    validation: Dataset({
        features: ['words', 'labels', 'is_category', 'text'],
        num_rows: 7951
    })
    test: Dataset({
        features: ['words', 'labels', 'is_category', 'text'],
        num_rows: 6844
    })
})


## 3. Exploratory analysis

Before modeling: inspect the schema, the label distribution (heavily imbalanced — `O` dominates ~85%), and verify data hygiene (words/labels alignment, sequence lengths).

In [2]:
# --- Schema: what one example looks like ---
sample = ds["train"][0]

for key in ("words", "labels", "text"):
    print(f"========== {key.upper()} ==========")
    print(sample[key])
    print()


========== WORDS ==========
['B', '.', 'Yeast', 'two', '-', 'hybrid', 'analysis', 'of', 'the', 'interactions', 'between', 'FREE1', 'and', 'core', 'microprocessor', '.', 'SD', '-', '2', ':', 'SD', '/', '-', 'Leu', '-', 'Trp', '.', 'SD', '-', '4', ':', 'SD', '/', '-', 'Ade', '-', 'His', '-', 'Leu', '-', 'Trp', '.', 'Three', 'dots', 'represent', 'three', 'yeast', 'colonies', 'randomly', 'picked', 'out', 'in', 'each', 'sample', '.']

========== LABELS ==========
['O', 'O', 'B-EXP_ASSAY', 'I-EXP_ASSAY', 'I-EXP_ASSAY', 'I-EXP_ASSAY', 'O', 'O', 'O', 'B-EXP_ASSAY', 'O', 'B-GENEPROD', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-ORGANISM', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']

========== TEXT ==========
B. Yeast two-hybrid analysis of the interactions between FREE1 and core microprocessor. SD-2: SD/-Leu-Trp. SD-4: SD/-Ade-His-Leu-Trp. Three dots represent three yeas

In [3]:
# --- Label distribution: O dominates; DISEASE is the rarest entity ---
from collections import Counter

label_counts = Counter(label for labels in ds["train"]["labels"] for label in labels)

for label, count in label_counts.most_common():
    print(f"{label:25} {count:,}")


O                         3,656,306
B-GENEPROD                201,476
B-EXP_ASSAY               84,804
I-EXP_ASSAY               57,819
B-SMALL_MOLECULE          57,419
I-GENEPROD                47,599
B-ORGANISM                30,193
B-SUBCELLULAR             27,688
B-TISSUE                  26,846
B-CELL_TYPE               21,519
I-SMALL_MOLECULE          20,898
B-CELL_LINE               19,870
I-CELL_LINE               10,352
I-ORGANISM                8,822
I-CELL_TYPE               8,164
I-SUBCELLULAR             7,965
I-TISSUE                  7,931
B-DISEASE                 5,059
I-DISEASE                 1,904


In [4]:
# --- Entity-type distribution (B- + I- merged): 9 entity types ---
entity_counts = Counter()

for labels in ds["train"]["labels"]:
    for label in labels:
        if label != "O":
            entity_counts[label.split("-", 1)[1]] += 1

for entity, count in entity_counts.most_common():
    print(f"{entity:20} {count:,}")


GENEPROD             249,075
EXP_ASSAY            142,623
SMALL_MOLECULE       78,317
ORGANISM             39,015
SUBCELLULAR          35,653
TISSUE               34,777
CELL_LINE            30,222
CELL_TYPE            29,683
DISEASE              6,963


In [5]:
# --- Hygiene checks ---
import numpy as np

# (a) words and labels must be aligned 1:1 in every example
for split in ("train", "validation", "test"):
    mismatches = sum(
        len(ex["words"]) != len(ex["labels"]) for ex in ds[split]
    )
    print(f"{split}: words/labels length mismatches = {mismatches}")

# (b) tokenized length distribution -> informs max_length (note: ~1% of train
# examples exceed 256 subword tokens; see §11 for the recommendation to raise it)
word_lengths = [len(ex["words"]) for ex in ds["train"]]
print("word-count percentiles [50/90/95/99]:",
      np.percentile(word_lengths, [50, 90, 95, 99]), "| max:", max(word_lengths))


train: words/labels length mismatches = 0
validation: words/labels length mismatches = 0
test: words/labels length mismatches = 0
word-count percentiles [50/90/95/99]: [ 66.   140.   171.   254.51] | max: 2651


## 4. The BIO tagging scheme

Each word carries one of three kinds of labels:

- **`O`** — outside any entity.
- **`B-<type>`** — first word of a new entity (e.g. `Yeast → B-EXP_ASSAY`).
- **`I-<type>`** — continuation of the entity that is currently open.

So a contiguous run `B-EXP_ASSAY, I-EXP_ASSAY, I-EXP_ASSAY` is **one** entity of type `EXP_ASSAY`.
Scoring is done entity-by-entity (seqeval, strict IOB2), not token-by-token — a prediction that gets
the right span but wrong B/I boundary counts as wrong.

In [6]:
def extract_entities(words, labels):
    """Rebuild entity spans {(text, type)} from a BIO-tagged sequence."""
    entities = []
    current_entity = None

    for word, label in zip(words, labels):
        if label == "O":
            if current_entity is not None:
                entities.append(current_entity)
                current_entity = None

        elif label.startswith("B-"):
            if current_entity is not None:
                entities.append(current_entity)
            current_entity = {"text": word, "type": label[2:]}

        elif label.startswith("I-"):
            # continuation of the currently open entity
            if current_entity is not None:
                current_entity["text"] += " " + word

    if current_entity is not None:
        entities.append(current_entity)

    return entities


# quick check on the first training example
for entity in extract_entities(sample["words"], sample["labels"]):
    print(entity)


{'text': 'Yeast two - hybrid', 'type': 'EXP_ASSAY'}
{'text': 'interactions', 'type': 'EXP_ASSAY'}
{'text': 'FREE1', 'type': 'GENEPROD'}
{'text': 'yeast', 'type': 'ORGANISM'}


In [7]:
# --- Color-coded visual inspection (helps spot annotation noise) ---
from IPython.display import display, HTML
import html

# one stable color per entity type
COLORS = {
    "DISEASE": "#ffcdd2", "EXP_ASSAY": "#bbdefb", "GENEPROD": "#c8e6c9",
    "SMALL_MOLECULE": "#ffe0b2", "ORGANISM": "#e1bee7", "SUBCELLULAR": "#d1c4e9",
    "TISSUE": "#fff9c4", "CELL_LINE": "#b2dfdb", "CELL_TYPE": "#f8bbd0",
}

def visualize_sample(dataset, idx):
    """Render sample `idx` as HTML with entity spans highlighted."""
    words, labels = dataset[idx]["words"], dataset[idx]["labels"]

    # group tokens into spans: entity runs vs plain tokens
    spans, current_words, current_type = [], [], None
    for word, label in zip(words, labels):
        if label.startswith("B-"):
            if current_words:
                spans.append((current_words, current_type))
            current_words, current_type = [word], label[2:]
        elif label.startswith("I-") and current_type == label[2:]:
            current_words.append(word)
        else:
            if current_words:
                spans.append((current_words, current_type))
            current_words, current_type = [], None
            spans.append(([word], None))
    if current_words:
        spans.append((current_words, current_type))

    rendered = []
    for span_words, entity in spans:
        text = html.escape(" ".join(span_words))
        if entity:
            color = COLORS.get(entity, "#eeeeee")
            rendered.append(
                f'<span style="background:{color}; padding:3px 5px; border-radius:4px;" '
                f'title="{html.escape(entity)}">{text} <b>[{html.escape(entity)}]</b></span>'
            )
        else:
            rendered.append(text)

    display(HTML(f'<div style="line-height:2.2; font-size:15px;">{" ".join(rendered)}</div>'))
    print(f"Sample index: {idx}")


import random
random.seed(42)
for idx in random.sample(range(len(ds["validation"])), 3):
    print("=" * 80)
    visualize_sample(ds["validation"], idx)


Sample index: 5238


Sample index: 912


Sample index: 204


## 5. Label vocabulary

Built from the **train** split only, after repairing BIO consistency: an `I-<type>` with no preceding
`B-<type>`/`I-<type>` of the same type is promoted to `B-<type>` (orphan repair). The public splits happen
to be clean (0 invalid transitions), but the repair is cheap insurance and matters if the data is ever
re-exported.

In [8]:
def fix_orphan_inside_tags(example):
    """Promote orphan I-<type> (no open entity of that type) to B-<type>."""
    fixed, previous = [], "O"
    for label in example["labels"]:
        if label.startswith("I-"):
            etype = label[2:]
            if previous not in (f"B-{etype}", f"I-{etype}"):
                label = f"B-{etype}"  # orphan -> promote
        fixed.append(label)
        previous = label
    example["labels"] = fixed
    return example


ds = ds.map(fix_orphan_inside_tags)

# sanity check: count remaining invalid BIO transitions across all splits
invalid = 0
for split in ds:
    for labels in ds[split]["labels"]:
        prev = "O"
        for label in labels:
            if label.startswith("I-") and prev not in (f"B-{label[2:]}", f"I-{label[2:]}"):
                invalid += 1
            prev = label
print("Remaining invalid BIO transitions:", invalid)


Map:   0%|          | 0/55250 [00:00<?, ? examples/s]

Map:   0%|          | 0/7951 [00:00<?, ? examples/s]

Map:   0%|          | 0/6844 [00:00<?, ? examples/s]

Remaining invalid BIO transitions: 0


In [9]:
label_list = sorted(label_counts.keys())          # fixed labels == observed ones here
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

print("Number of labels:", len(label_list))
print(label2id)


Number of labels: 19
{'B-CELL_LINE': 0, 'B-CELL_TYPE': 1, 'B-DISEASE': 2, 'B-EXP_ASSAY': 3, 'B-GENEPROD': 4, 'B-ORGANISM': 5, 'B-SMALL_MOLECULE': 6, 'B-SUBCELLULAR': 7, 'B-TISSUE': 8, 'I-CELL_LINE': 9, 'I-CELL_TYPE': 10, 'I-DISEASE': 11, 'I-EXP_ASSAY': 12, 'I-GENEPROD': 13, 'I-ORGANISM': 14, 'I-SMALL_MOLECULE': 15, 'I-SUBCELLULAR': 16, 'I-TISSUE': 17, 'O': 18}


## 6. Tokenization and label alignment

BioBERT is a subword tokenizer, so words split (e.g. `microprocessor → micro ##p ##ro ##cess ##or`).
We must decide what label continuation subwords get. Two schemes were tried (see §11):

1. **Propagation** — continuation subwords inherit the label (`B-` → `I-`). Val F1 **0.8323**.
2. **`-100` masking** (implemented below) — only the *first* subword of each word is labeled;
   everything else (special tokens + continuations) is masked out of the loss. Val F1 **0.8391** ✅

Masking won: each word is scored exactly once, so entity boundaries line up with the word-level
gold annotations, and the model doesn't waste capacity reconciling subword-level B/I decisions.

In [10]:
from transformers import AutoTokenizer

model_checkpoint = "dmis-lab/biobert-base-cased-v1.2"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, use_fast=True)


def tokenize_and_align_labels(examples):
    """Subword-tokenize and align labels: first subword gets the label, rest get -100."""
    tokenized = tokenizer(
        examples["words"],
        is_split_into_words=True,
        truncation=True,
        max_length=256,
    )

    all_labels = []
    for i, labels in enumerate(examples["labels"]):
        word_ids = tokenized.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:                        # [CLS] / [SEP] / padding
                label_ids.append(-100)
            elif word_idx != previous_word_idx:         # first subword of a word
                label_ids.append(label2id[labels[word_idx]])
            else:                                       # continuation subword
                label_ids.append(-100)
            previous_word_idx = word_idx

        all_labels.append(label_ids)

    tokenized["labels"] = all_labels
    return tokenized


tokenized_ds = ds.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=ds["train"].column_names,
)


config.json:   0%|          | 0.00/1.11k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

Map:   0%|          | 0/55250 [00:00<?, ? examples/s]

Map:   0%|          | 0/7951 [00:00<?, ? examples/s]

Map:   0%|          | 0/6844 [00:00<?, ? examples/s]

## 7. Model, collator, metrics

Standard HF setup: `BertForTokenClassification` with a 19-way linear head on top of BioBERT.
The head weights (`classifier.*`) are randomly initialized — the "MISSING" keys in the load report are
expected and correct.

Metrics are **entity-level** (seqeval, strict IOB2): a span must match boundaries *and* type to count.

In [11]:
from transformers import AutoModelForTokenClassification, DataCollatorForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
)

# pads input_ids/labels to batch max; label padding uses -100 (ignored by the loss)
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)


[transformers] You passed `num_labels=19` which is incompatible to the `id2label` map of length `2`.


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  436MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dmis-lab/biobert-base-cased-v1.2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	

In [13]:
!pip install -U pip setuptools wheel
!pip install evaluate seqeval

  Using cached evaluate-0.4.6-py3-none-any.whl.metadata (9.5 kB)
  Using cached seqeval-1.2.2.tar.gz (43 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16251 sha256=5d5c67bee756b61ad8f053c5fcd6e168608b8a8708b88f506e3fbbcad52af718
  Stored in directory: /root/.cache/pip/wheels/14/cf/a7/8f28ef376d707ff10e3922899482a2f23ef3002f8a952f47ac
Successfully built seqeval
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [evaluate]


In [14]:
import numpy as np
import evaluate

seqeval = evaluate.load("seqeval")


def compute_metrics(eval_pred):
    """Entity-level P/R/F1 (strict IOB2). Ignores -100 positions."""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    true_predictions, true_labels = [], []
    for prediction, label in zip(predictions, labels):
        current_preds, current_labels = [], []
        for pred_id, label_id in zip(prediction, label):
            if label_id == -100:
                continue
            current_preds.append(id2label[pred_id])
            current_labels.append(id2label[label_id])
        true_predictions.append(current_preds)
        true_labels.append(current_labels)

    results = seqeval.compute(
        predictions=true_predictions,
        references=true_labels,
        scheme="IOB2",
        mode="strict",
    )
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }


## 8. Training

- batch 16, lr 2e-5, linear schedule with 10% warmup, weight decay 0.01, fp16, 3 epochs
- checkpoint selection by **val F1** (`load_best_model_at_end`), early stopping patience 2
- ~55k train examples → 3,453 steps/epoch, 10,362 total (≈1 h on a T4)

In [15]:
from transformers import TrainingArguments

batch_size = 16
num_epochs = 3

training_args = TrainingArguments(
    output_dir="./biobert_sourcedata_ner",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=num_epochs,
    weight_decay=0.01,
    warmup_steps=int(0.1 * (len(tokenized_ds["train"]) // batch_size) * num_epochs),
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    fp16=True,
    logging_steps=100,
    report_to="none",
)


In [16]:
from transformers import Trainer, EarlyStoppingCallback

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.102541,0.132502,0.794065,0.840398,0.816574,0.957481
2,0.075161,0.125713,0.814287,0.844667,0.829199,0.961055
3,0.058365,0.128583,0.823173,0.848814,0.835797,0.962091


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=10362, training_loss=0.12284795560562525, metrics={'train_runtime': 2017.0873, 'train_samples_per_second': 82.173, 'train_steps_per_second': 5.137, 'total_flos': 1.905629393998649e+16, 'train_loss': 0.12284795560562525, 'epoch': 3.0})

## 9. Save

`save_model` writes config + weights + the head's label maps; tokenizer files are saved alongside for self-contained inference.

In [17]:
trainer.save_model("./biobert_sourcedata_ner/final")
tokenizer.save_pretrained("./biobert_sourcedata_ner/final")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./biobert_sourcedata_ner/final/tokenizer_config.json',
 './biobert_sourcedata_ner/final/tokenizer.json')

## 10. Final evaluation on the **test** split

Validation F1 is used for checkpoint selection, so reporting it as the final number is mildly optimistic.
The held-out test set gives the unbiased estimate — this cell was missing from the original notebook.

In [18]:
test_logits, test_labels, _ = trainer.predict(tokenized_ds["test"])

# strip -100 and decode to label strings
pred_seqs, gold_seqs = [], []
for pred_row, label_row in zip(np.argmax(test_logits, axis=-1), test_labels):
    preds, golds = [], []
    for pred_id, label_id in zip(pred_row, label_row):
        if label_id == -100:
            continue
        preds.append(id2label[pred_id])
        golds.append(id2label[label_id])
    pred_seqs.append(preds)
    gold_seqs.append(golds)

test_results = seqeval.compute(predictions=pred_seqs, references=gold_seqs,
                               scheme="IOB2", mode="strict")

print("TEST  P:", round(test_results["overall_precision"], 4),
      "| R:", round(test_results["overall_recall"], 4),
      "| F1:", round(test_results["overall_f1"], 4))

# per-entity breakdown, worst-first -> tells you where to target next
import pandas as pd
rows = [
    {"entity": k, "precision": round(v["precision"], 4), "recall": round(v["recall"], 4),
     "f1": round(v["f1"], 4), "support": v["number"]}
    for k, v in test_results.items()
    if isinstance(v, dict)
]
per_entity = pd.DataFrame(rows).sort_values("f1")
print(per_entity.to_string(index=False))


TEST  P: 0.8277 | R: 0.8411 | F1: 0.8343
        entity  precision  recall     f1  support
       DISEASE     0.6348  0.6773 0.6553      598
     EXP_ASSAY     0.6961  0.6950 0.6955    10075
     CELL_TYPE     0.7039  0.7347 0.7190     2782
   SUBCELLULAR     0.7794  0.7740 0.7767     4154
        TISSUE     0.8087  0.8429 0.8254     3621
SMALL_MOLECULE     0.8394  0.8260 0.8327     6678
      ORGANISM     0.8656  0.9003 0.8826     3792
     CELL_LINE     0.8793  0.9054 0.8922     2358
      GENEPROD     0.8934  0.9137 0.9034    25654


## 11. What was tried, and what to try next

**Ablations already run (validation F1, strict IOB2):**
| change | val F1 | verdict |
|---|---|---|
| continuation subwords get `I-` | 0.8323 | ❌ worse |
| continuation subwords masked `-100` | 0.8391 | ✅ kept |
| CRF head + Dice loss (dice_weight 0.5), batch 16 | 0.8373 | ≈ tied |

**Why the CRF+Dice "huge loss" (~7.7 → 4.4) is not a problem:** the CRF objective is
`-log P(y|x) = -(emission_score - log Z(x))` — it includes the partition function `Z(x)`, a sum over
**all possible label sequences**, so its scale is unrelated to cross-entropy (~0.1–0.2). A CRF NLL of
5–10 per sequence is completely normal; only its *trend* matters, and it decreased every epoch while F1
rose (0.817 → 0.831 → 0.837). Note the CRF run's validation F1 was argmax-based (no Viterbi), so it
slightly *understates* the CRF model — the fair comparison is the Viterbi-decoded **test** F1 from the
CRF notebook vs. the test F1 in §10 here.

**Next levers, in expected impact order:**
1. **`max_length` 256 → 384.** ~1% of examples are truncated at 256; long biomedical sentences are
   exactly where multi-word entities live.
2. **Ablate the Dice term** (`dice_weight=0`, pure CRF). CRF NLL already encodes structure; Dice may be
   redundant — if CRF-only matches CRF+Dice, drop Dice for a simpler model.
3. **Discriminative learning rates** — encoder 2e-5 but classifier/CRF head 5e-4; the head is random-init
   and can move faster.
4. **Weight the CE loss by inverse label frequency** — `O` is ~85% of tokens; mild weighting on `B-/I-`
   tags often buys a point of recall on rare types (DISEASE has only ~7k train mentions).
5. **Seeds + span-friendly error analysis** — check the per-entity table from §10; GENEPROD boundary
   errors vs. EXP_ASSAY confusion are the usual win sources.

In [19]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [20]:
import shutil

src = "/content/biobert_sourcedata_ner"
dst = "/content/drive/MyDrive/biobert_sourcedata_ner"

shutil.copytree(src, dst, dirs_exist_ok=True)

print("Folder copied successfully!")

Folder copied successfully!


In [21]:
import os

print(os.listdir("/content/drive/MyDrive/biobert_sourcedata_ner"))

['checkpoint-6908', 'checkpoint-10362', 'final']
